In [ ]:
import os
import shutil
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets, transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from torchvision.models import efficientnet_b2, EfficientNet_B2_Weights
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import csv
import warnings
warnings.filterwarnings("ignore")

KVASIR_ROOT    = "/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset"
SYNTHETIC_ROOT = "/kaggle/input/datasets/yumna19/endoscopy-medical-image-generation/generated_images"
OUTPUT_ROOT    = "/kaggle/working/split_dataset"

IMG_SIZE          = 224
BATCH_SIZE        = 32
EPOCHS            = 90
SYNTH_TRAIN_RATIO = 1.0       
LR                = 1e-4
WEIGHT_DECAY      = 1e-3
LABEL_SMOOTHING   = 0.03
N_SPLITS          = 5
TEST_SPLIT        = 0.15
RANDOM_SEED       = 42
NUM_WORKERS       = 2
PATIENCE          = 15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}

def collect_image_paths(root_dir):
    samples = []
    classes = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])
    print(f"\nClasses found in '{root_dir}': {classes}")

    for cls in classes:
        cls_dir = os.path.join(root_dir, cls)
        sub = os.listdir(cls_dir)
        if len(sub) == 1 and os.path.isdir(os.path.join(cls_dir, sub[0])):
            cls_dir = os.path.join(cls_dir, sub[0])
        for fname in sorted(os.listdir(cls_dir)):
            if Path(fname).suffix.lower() in VALID_EXTS:
                samples.append((os.path.join(cls_dir, fname), cls))

    print(f"Total images: {len(samples)}")
    return samples, classes

def copy_files(samples, split_name, base_dir):
    for src, cls in samples:
        dst_dir = os.path.join(base_dir, split_name, cls)
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy2(src, dst_dir)


def create_splits(kvasir_root, synthetic_root, output_root):
    real_samples, real_classes = collect_image_paths(kvasir_root)

    real_trainval, real_test = train_test_split(
        real_samples, test_size=TEST_SPLIT,
        random_state=RANDOM_SEED,
        stratify=[s[1] for s in real_samples]
    )

    synth_samples, synth_classes = collect_image_paths(synthetic_root)
    assert set(real_classes) == set(synth_classes), (
        f"Class mismatch!\n  Real : {real_classes}\n  Synth: {synth_classes}"
    )

    synth_train = synth_samples

    print(f"\n── Data split ──")
    print(f"  Real train+val : {len(real_trainval)}")
    print(f"  Real test      : {len(real_test)}")
    print(f"  Synthetic train: {len(synth_train)}")

    copy_files(real_trainval, "real_train", output_root)
    copy_files(synth_train,   "synthetic",  output_root)
    copy_files(real_test,     "test",       output_root)

    return real_classes, len(real_trainval), len(synth_train)



_MEAN = [0.485, 0.456, 0.406]
_STD  = [0.229, 0.224, 0.225]

def get_transform(size=224):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(_MEAN, _STD),
    ])



class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.03):
        super().__init__()
        self.weight          = weight
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )(inputs, targets)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()



MODEL_REGISTRY = {}

def _register(name):
    def decorator(fn):
        MODEL_REGISTRY[name] = fn
        return fn
    return decorator


@_register("efficientnet_b0")
def build_efficientnet_b0(num_classes):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

    for p in model.parameters():
        p.requires_grad = False

    children = list(model.features.children())
    for block in children[-4:]:
        for p in block.parameters():
            p.requires_grad = True

    in_f = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.40, inplace=False),
        nn.Linear(in_f, 512),
        nn.LeakyReLU(negative_slope=0.01, inplace=False),
        nn.BatchNorm1d(512),
        nn.Dropout(p=0.30, inplace=False),
        nn.Linear(512, num_classes),
    )
    _print_param_count(model, "EfficientNet-B0")
    return model.to(DEVICE)


@_register("efficientnet_b2")
def build_efficientnet_b2(num_classes):
    model = efficientnet_b2(weights=EfficientNet_B2_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    for block in list(model.features.children())[-5:]:
        for p in block.parameters():
            p.requires_grad = True
    in_f = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.35),
        nn.Linear(in_f, 512),
        nn.LeakyReLU(0.01),
        nn.BatchNorm1d(512),
        nn.Dropout(p=0.25),
        nn.Linear(512, num_classes),
    )
    _print_param_count(model, "EfficientNet-B2")
    return model.to(DEVICE)


@_register("efficientnet_b3")
def build_efficientnet_b3(num_classes):
    model = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)

    for p in model.parameters():
        p.requires_grad = False



    children = list(model.features.children())
    for block in children[-3:]:
        for p in block.parameters():
            p.requires_grad = True

    in_f = model.classifier[1].in_features   
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.50, inplace=False),
        nn.Linear(in_f, 768),
        nn.LeakyReLU(negative_slope=0.01, inplace=False),
        nn.BatchNorm1d(768),
        nn.Dropout(p=0.40, inplace=False),
        nn.Linear(768, 256),
        nn.LeakyReLU(negative_slope=0.01, inplace=False),
        nn.BatchNorm1d(256),
        nn.Dropout(p=0.25, inplace=False),
        nn.Linear(256, num_classes),
    )

    _print_param_count(model, "EfficientNet-B3")
    return model.to(DEVICE)


def _print_param_count(model, name):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  [{name}] Trainable: {trainable:>10,} / {total:>10,} "
          f"({100 * trainable / total:.1f}%)")

 
MODEL_INPUT_SIZES = {
    "efficientnet_b0": 224,
    "efficientnet_b2": 260,
    "efficientnet_b3": 300,
}

MODEL_DISPLAY_NAMES = {
    "efficientnet_b0": "EfficientNet-B0",
    "efficientnet_b2": "EfficientNet-B2",
    "efficientnet_b3": "EfficientNet-B3",
}




def train_one_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = out.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, labels)
        running_loss += loss.item() * imgs.size(0)
        probs = torch.softmax(out, dim=1)
        _, preds = probs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
    all_probs = np.vstack(all_probs)
    return running_loss / total, correct / total, all_preds, all_labels, all_probs





def run_kfold_for_model(model_name, output_root, num_classes, class_names):
    img_size = MODEL_INPUT_SIZES[model_name]
    tf       = get_transform(img_size)

    real_train_ds = datasets.ImageFolder(
        os.path.join(output_root, "real_train"), transform=tf
    )


    synth_folder = "synthetic_filtered"
    synth_path   = os.path.join(output_root, synth_folder)
    if not os.path.isdir(synth_path):
        print(f"  [WARNING] '{synth_folder}' not found, falling back to 'synthetic'")
        synth_folder = "synthetic"
        synth_path   = os.path.join(output_root, synth_folder)

    synthetic_ds = datasets.ImageFolder(synth_path, transform=tf)

    assert real_train_ds.class_to_idx == synthetic_ds.class_to_idx, (
        f"[{model_name}] Class index mismatch!\n"
        f"  Real : {real_train_ds.class_to_idx}\n"
        f"  Synth: {synthetic_ds.class_to_idx}"
    )

    indices = np.arange(len(real_train_ds))
    labels  = np.array(real_train_ds.targets)
    skf     = StratifiedKFold(n_splits=N_SPLITS, shuffle=True,
                               random_state=RANDOM_SEED)
    fold_results = []

    print(f"\n{'='*65}")
    print(f"  MODEL: {MODEL_DISPLAY_NAMES[model_name]}  "
          f"(input {img_size}×{img_size})")
    print(f"  Real train: {len(real_train_ds)}  |  "
          f"Synthetic ({synth_folder}): {len(synthetic_ds)}  |  Folds: {N_SPLITS}")
    print(f"  Loss: FocalLoss(gamma=2.0) + label_smoothing={LABEL_SMOOTHING}")
    print(f"{'='*65}")

    os.makedirs("checkpoints", exist_ok=True)

    for fold, (train_idx, val_idx) in enumerate(
            skf.split(indices, labels), 1):

        print(f"\n  ── Fold {fold}/{N_SPLITS} ──")

        real_len = len(train_idx)


        synth_indices  = np.arange(len(synthetic_ds))
        n_synth_sample = len(synth_indices)

        synth_subset   = Subset(synthetic_ds, synth_indices)
        train_combined = ConcatDataset([
            Subset(real_train_ds, train_idx),
            synth_subset,
        ])
        val_subset = Subset(real_train_ds, val_idx)

        print(f"     Train fold : {real_len}  +  "
              f"{n_synth_sample} synthetic = "
              f"{len(train_combined)} total")
        print(f"     Val fold   : {len(val_idx)} (real only)")

        train_loader = DataLoader(
            train_combined, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True
        )
        val_loader = DataLoader(
            val_subset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True
        )

        model = MODEL_REGISTRY[model_name](num_classes)


        label_arr     = np.array(real_train_ds.targets)[train_idx]
        class_counts  = np.bincount(label_arr, minlength=num_classes).astype(float)
        class_weights = 1.0 / (class_counts + 1e-6)
 
        for cls_name in ["esophagitis", "normal-z-line"]:
            if cls_name in real_train_ds.class_to_idx:
                idx = real_train_ds.class_to_idx[cls_name]
                class_weights[idx] *= 2.0

        for cls_name in ["dyed-lifted-polyps", "dyed-resection-margins"]:
            if cls_name in real_train_ds.class_to_idx:
                idx = real_train_ds.class_to_idx[cls_name]
                class_weights[idx] *= 1.5

        class_weights = class_weights / class_weights.sum() * num_classes
        weight_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)


        criterion = FocalLoss(
            weight=weight_tensor,
            gamma=2.0,
            label_smoothing=LABEL_SMOOTHING,
        )

        feature_children     = list(model.features.named_children())
        block_names_by_depth = [fc[0] for fc in feature_children]

        classifier_params = []
        last1_params      = []  
        last2_3_params    = []   

        last1   = block_names_by_depth[-1]
        last2_3 = {block_names_by_depth[-2], block_names_by_depth[-3]}

        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if "classifier" in name:
                classifier_params.append(p)
            elif f"features.{last1}." in name:
                last1_params.append(p)
            else:
                last2_3_params.append(p)

        optimizer = optim.AdamW([
            {"params": last2_3_params,    "lr": LR * 0.03, "weight_decay": WEIGHT_DECAY * 1.5},
            {"params": last1_params,      "lr": LR * 0.07, "weight_decay": WEIGHT_DECAY * 2.0},
            {"params": classifier_params, "lr": LR,        "weight_decay": WEIGHT_DECAY * 4.0},
        ])

        scheduler = optim.lr_scheduler.OneCycleLR(
            optimizer,
            max_lr=[LR * 0.03, LR * 0.07, LR],   
            steps_per_epoch=len(train_loader),
            epochs=EPOCHS,
            pct_start=0.30,
            anneal_strategy='cos',
            div_factor=10.0,
            final_div_factor=500.0,
        )

        best_val_acc, best_state = 0.0, None
        patience_ctr = 0
        history = {k: [] for k in
                   ["train_loss", "train_acc", "val_loss", "val_acc"]}

        for epoch in range(1, EPOCHS + 1):
            t_loss, t_acc = train_one_epoch(
                model, train_loader, optimizer, scheduler, criterion
            )
            v_loss, v_acc, _, _, _ = evaluate(
                model, val_loader, criterion
            )

            history["train_loss"].append(t_loss)
            history["train_acc"].append(t_acc)
            history["val_loss"].append(v_loss)
            history["val_acc"].append(v_acc)

            print(f"     Epoch {epoch:02d}/{EPOCHS}  "
                  f"Train {t_acc:.4f} ({t_loss:.4f})  |  "
                  f"Val {v_acc:.4f} ({v_loss:.4f})")

            if v_acc > best_val_acc:
                best_val_acc = v_acc
                best_state   = {k: v.cpu().clone()
                                for k, v in model.state_dict().items()}
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"     Early stop at epoch {epoch}")
                    break

        torch.save(best_state,
                   f"checkpoints/{model_name}_fold{fold}.pth")

        fold_results.append({
            "fold":         fold,
            "best_val_acc": best_val_acc,
            "history":      history,
        })
        print(f"\n  ✓ Fold {fold}  best val acc: {best_val_acc*100:.2f}%")

    return fold_results



@torch.no_grad()
def _collect_probs(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs  = imgs.to(DEVICE)
        out   = model(imgs)
        probs = torch.softmax(out, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.extend(labels.numpy())
    return np.vstack(all_probs), np.array(all_labels)


def evaluate_model_on_test(model_name, output_root, num_classes, class_names):
    img_size    = MODEL_INPUT_SIZES[model_name]
    tf          = get_transform(img_size)
    test_ds     = datasets.ImageFolder(
        os.path.join(output_root, "test"), transform=tf
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True
    )

    prob_sum = None
    labels   = None

    for fold in range(1, N_SPLITS + 1):
        ckpt  = f"checkpoints/{model_name}_fold{fold}.pth"
        model = MODEL_REGISTRY[model_name](num_classes)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        probs, lbls = _collect_probs(model, test_loader)
        prob_sum = probs if prob_sum is None else prob_sum + probs
        labels   = lbls
        fold_acc = accuracy_score(lbls, probs.argmax(1))
        print(f"  [{MODEL_DISPLAY_NAMES[model_name]}] "
              f"Fold {fold} test acc: {fold_acc*100:.2f}%")

    avg_probs   = prob_sum / N_SPLITS
    final_preds = avg_probs.argmax(axis=1)
    acc = accuracy_score(labels, final_preds)
    f1  = f1_score(labels, final_preds, average="weighted")

    try:
        auc = roc_auc_score(labels, avg_probs, multi_class="ovr",
                            average="weighted")
    except Exception:
        auc = float("nan")

    print(f"\n  [{MODEL_DISPLAY_NAMES[model_name]}] "
          f"Soft-Ensemble  Acc={acc*100:.2f}%  "
          f"F1={f1:.4f}  AUC={auc:.4f}")
    print(classification_report(labels, final_preds,
                                 target_names=class_names))

    _plot_confusion_matrix(
        labels, final_preds, class_names,
        title=f"{MODEL_DISPLAY_NAMES[model_name]}  —  Acc={acc*100:.2f}%",
        filename=f"cm_{model_name}.png"
    )

    return {
        "model":       model_name,
        "accuracy":    acc,
        "f1_weighted": f1,
        "auc_ovr":     auc,
        "preds":       final_preds,
        "labels":      labels,
    }


# ═══════════════════════════════════════════════════════════════════
#  HELPERS — plotting
# ═══════════════════════════════════════════════════════════════════

def _plot_confusion_matrix(labels, preds, class_names,
                            title, filename, cmap="Blues"):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(max(8, len(class_names)), 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.5)
    ax.set_title(title, fontsize=12)
    ax.set_ylabel("True label")
    ax.set_xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"  Saved → {filename}")


def plot_training_curves(fold_results, model_name):
    display = MODEL_DISPLAY_NAMES[model_name]
    fig, axes = plt.subplots(2, N_SPLITS, figsize=(5 * N_SPLITS, 8))
    for i, fr in enumerate(fold_results):
        h = fr["history"]
        axes[0, i].plot(h["train_loss"], label="Train")
        axes[0, i].plot(h["val_loss"],   label="Val")
        axes[0, i].set_title(f"Fold {fr['fold']} – Loss")
        axes[0, i].legend()
        axes[1, i].plot(h["train_acc"], label="Train")
        axes[1, i].plot(h["val_acc"],   label="Val")
        axes[1, i].set_title(f"Fold {fr['fold']} – Acc")
        axes[1, i].legend()
    plt.suptitle(f"{display} — K-Fold Training Curves", fontsize=13)
    plt.tight_layout()
    fname = f"curves_{model_name}.png"
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"  Saved → {fname}")




def save_all_metrics(fold_results, result):
    model_name = result["model"]

    with open("training_log.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["fold", "epoch",
                    "train_loss", "train_acc", "val_loss", "val_acc"])
        for fr in fold_results:
            h = fr["history"]
            for ep in range(len(h["train_loss"])):
                w.writerow([
                    fr["fold"], ep + 1,
                    round(h["train_loss"][ep], 4),
                    round(h["train_acc"][ep],  4),
                    round(h["val_loss"][ep],   4),
                    round(h["val_acc"][ep],    4),
                ])
    print("Saved → training_log.csv")

    with open("model_test_summary.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["model", "accuracy", "f1_weighted", "auc_ovr"])
        w.writerow([
            MODEL_DISPLAY_NAMES[model_name],
            round(result["accuracy"],    4),
            round(result["f1_weighted"], 4),
            round(result["auc_ovr"],     4) if not np.isnan(result["auc_ovr"]) else "N/A",
        ])
    print("Saved → model_test_summary.csv")

    summary = {
        "model":           MODEL_DISPLAY_NAMES[model_name],
        "accuracy":        float(result["accuracy"]),
        "f1_weighted":     float(result["f1_weighted"]),
        "auc_ovr":         float(result["auc_ovr"]) if not np.isnan(result["auc_ovr"]) else None,
        "kfold_val_accs":  [fr["best_val_acc"] for fr in fold_results],
        "kfold_val_mean":  float(np.mean([fr["best_val_acc"] for fr in fold_results])),
        "kfold_val_std":   float(np.std([fr["best_val_acc"] for fr in fold_results])),
    }
    with open("results_summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print("Saved → results_summary.json")


def save_final_accuracy_csv(fold_results, result):
    model_name     = result["model"]
    mean_train_acc = np.mean([fr["history"]["train_acc"][-1] for fr in fold_results])
    mean_val_acc   = np.mean([fr["best_val_acc"] for fr in fold_results])
    test_acc       = result["accuracy"]

    with open("final_accuracy_summary.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["model", "mean_train_acc", "mean_val_acc", "test_acc"])
        w.writerow([
            MODEL_DISPLAY_NAMES[model_name],
            round(float(mean_train_acc), 4),
            round(float(mean_val_acc),   4),
            round(float(test_acc),       4),
        ])
    print("Saved → final_accuracy_summary.csv")
    print("\n  Final accuracy summary:")
    print(f"  {'Model':<22} {'Train':>10} {'Val':>10} {'Test':>10}")
    print(f"  {'-'*54}")
    print(f"  {MODEL_DISPLAY_NAMES[model_name]:<22} "
          f"{mean_train_acc*100:>9.2f}%"
          f"{mean_val_acc*100:>10.2f}%"
          f"{test_acc*100:>10.2f}%")


import cv2
from PIL import Image

def compute_image_quality_score(img_path):
    try:
        img_pil = Image.open(img_path).convert("RGB").resize((224, 224))
        img_np  = np.array(img_pil)
        gray      = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
        contrast  = img_np.std()
        hsv        = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)
        saturation = hsv[:, :, 1].mean()
        return {"path": img_path, "sharpness": sharpness,
                "contrast": contrast, "saturation": saturation}
    except Exception as e:
        print(f"  Warning: could not score {img_path}: {e}")
        return None


def filter_synthetic_by_quality(synthetic_root, output_root,
                                  class_names, keep_top_pct=0.70):
    print(f"\n── Synthetic quality filter (keeping top {keep_top_pct*100:.0f}%) ──")
    VALID_EXTS    = {".jpg", ".jpeg", ".png", ".bmp", ".tiff"}
    total_kept    = 0
    total_dropped = 0
    stats_log     = []

    for cls in class_names:
        cls_dir = os.path.join(synthetic_root, cls)
        if not os.path.isdir(cls_dir):
            continue
        sub = os.listdir(cls_dir)
        if len(sub) == 1 and os.path.isdir(os.path.join(cls_dir, sub[0])):
            cls_dir = os.path.join(cls_dir, sub[0])
        img_paths = [
            os.path.join(cls_dir, f)
            for f in sorted(os.listdir(cls_dir))
            if Path(f).suffix.lower() in VALID_EXTS
        ]
        scores = [s for s in
                  (compute_image_quality_score(p) for p in img_paths)
                  if s is not None]
        if not scores:
            print(f"  [{cls}] No images found, skipping.")
            continue

        def norm(arr):
            r = arr.max() - arr.min()
            return (arr - arr.min()) / r if r > 0 else np.zeros_like(arr)

        composite = (
            norm(np.array([s["sharpness"]  for s in scores])) +
            norm(np.array([s["contrast"]   for s in scores])) +
            norm(np.array([s["saturation"] for s in scores]))
        ) / 3.0

        n_keep  = max(1, int(len(scores) * keep_top_pct))
        top_idx = np.argsort(composite)[::-1][:n_keep]
        kept    = [scores[i]["path"] for i in top_idx]
        dropped = len(scores) - n_keep

        dst_dir = os.path.join(output_root, "synthetic_filtered", cls)
        os.makedirs(dst_dir, exist_ok=True)
        for src in kept:
            shutil.copy2(src, dst_dir)

        total_kept    += n_keep
        total_dropped += dropped
        stats_log.append({
            "class": cls, "original_count": len(scores),
            "kept": n_keep, "dropped": dropped,
            "mean_score_kept": composite[top_idx].mean(),
            "mean_score_all":  composite.mean(),
        })
        print(f"  [{cls}] {len(scores)} → kept {n_keep}, dropped {dropped}  "
              f"(mean score: {composite[top_idx].mean():.3f})")

    with open("synthetic_filter_report.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["class", "original", "kept", "dropped",
                    "mean_score_kept", "mean_score_all"])
        for r in stats_log:
            w.writerow([r["class"], r["original_count"], r["kept"],
                        r["dropped"], round(r["mean_score_kept"], 4),
                        round(r["mean_score_all"], 4)])

    print(f"\n  Total kept   : {total_kept}")
    print(f"  Total dropped: {total_dropped}")
    print(f"  Saved → synthetic_filter_report.csv")
    return os.path.join(output_root, "synthetic_filtered")


if __name__ == "__main__":

    print("\n── STEP 1: Creating splits ──")
    classes, n_real, n_synth = create_splits(
        KVASIR_ROOT, SYNTHETIC_ROOT, OUTPUT_ROOT
    )
    num_classes = len(classes)
    print(f"\nClasses ({num_classes}): {classes}")

 
    print("\n── STEP 1b: Filtering synthetic images by quality ──")
    filter_synthetic_by_quality(
        os.path.join(OUTPUT_ROOT, "synthetic"),
        OUTPUT_ROOT,
        classes,
        keep_top_pct=0.70,
    )

    print("\n── STEP 2: Stratified K-Fold training ──")
    MODEL_NAME   = "efficientnet_b3"
    fold_results = run_kfold_for_model(
        MODEL_NAME, OUTPUT_ROOT, num_classes, classes
    )
    kf_accs = [fr["best_val_acc"] for fr in fold_results]
    print(f"\n  [{MODEL_DISPLAY_NAMES[MODEL_NAME]}] "
          f"K-Fold Val: mean={np.mean(kf_accs)*100:.2f}%  "
          f"std={np.std(kf_accs)*100:.2f}%")

    print("\n── STEP 3: Test evaluation ──")
    result = evaluate_model_on_test(
        MODEL_NAME, OUTPUT_ROOT, num_classes, classes
    )

    print("\n── STEP 4: Plots and metrics ──")
    plot_training_curves(fold_results, MODEL_NAME)
    save_all_metrics(fold_results, result)
    save_final_accuracy_csv(fold_results, result)

    print("\n" + "="*65)
    print("  FINAL SUMMARY")
    print("="*65)
    print(f"\n  {MODEL_DISPLAY_NAMES[MODEL_NAME]}")
    print(f"    K-Fold Val Acc : {np.mean(kf_accs)*100:.2f}% ± {np.std(kf_accs)*100:.2f}%")
    print(f"    Test Accuracy  : {result['accuracy']*100:.2f}%")
    print(f"    F1 (weighted)  : {result['f1_weighted']:.4f}")
    print(f"    AUC OvR        : {result['auc_ovr']:.4f}")

